In [1]:
from pecos.slr import Main, SlrConverter, CReg, QReg
from pecos.qeclib.steane.steane_class import Steane

In [7]:
def steane_flagged_prep(basis: str = "Z", syn_rounds: int = 0, num_lqs: int = 1, ):

    s = [
        Steane(f"s_{i}", ancillas=QReg(f"s_a_{i}", 1), flag_qubits=QReg(f"s_f_{i}", 1)) 
               for i in range(num_lqs)
    ]

    
    init_reject = [CReg(f"init_reject_{i}", 1) for i in range(num_lqs)]
    
    log_out = [CReg(f"log_{i}", 1) for i in range(num_lqs)]

    syn = []
    for i in range(num_lqs):
        syn.append([CReg(f"syn_{i}_{j}", 6) for j in range(syn_rounds)])

    flags = []
    for i in range(num_lqs):
        flags.append([CReg(f"flags_{i}_{j}", 6) for j in range(syn_rounds)])
    
    
    
    prog = Main(
        *init_reject,
        *log_out,
    )

    for i in range(num_lqs):
        prog.extend(
            *syn[i],
            *flags[i],
        )

    prog.extend(
        s[0],
        s[0].p(state=basis, reject=init_reject[0][0], rus_limit=1),
    )
    
    for r in range(syn_rounds):
        prog.extend(
            s[0].syn_flagged(syn[0][r], flags[0][r])
        )
    
    prog.extend(
        
        # TODO: Measure raw
        s[0].m(meas_basis=basis, log=log_out[0][0]),
    )
    return prog

In [8]:
prog = steane_flagged_prep(syn_rounds=4)
qasm = SlrConverter(prog).qasm()

syn_0_0
syn_0_1
syn_0_2
syn_0_3


In [9]:
print(qasm)

OPENQASM 2.0;
include "hqslib1.inc";
creg init_reject_0[1];
creg log_0[1];
creg syn_0_0[6];
creg syn_0_1[6];
creg syn_0_2[6];
creg syn_0_3[6];
creg flags_0_0[6];
creg flags_0_1[6];
creg flags_0_2[6];
creg flags_0_3[6];
qreg s_0_d[7];
creg s_0_c[32];
creg s_0_syn_meas[32];
creg s_0_last_raw_syn_x[32];
creg s_0_last_raw_syn_z[32];
creg s_0_scratch[32];
creg s_0_flag_x[3];
creg s_0_flags_z[3];
creg s_0_flags[3];
creg s_0_raw_meas[7];
creg s_0_syn_x[3];
creg s_0_syn_z[3];
creg s_0_syndromes[3];
creg s_0_verify_prep[32];

barrier s_0_d[0], s_0_d[1], s_0_d[2], s_0_d[3], s_0_d[4], s_0_d[5], s_0_d[6], s_a_0[0];

reset s_0_d[0];
reset s_0_d[1];
reset s_0_d[2];
reset s_0_d[3];
reset s_0_d[4];
reset s_0_d[5];
reset s_0_d[6];
reset s_a_0[0];
barrier s_0_d, s_a_0[0];
h s_0_d[0];
h s_0_d[4];
h s_0_d[6];

cx s_0_d[4], s_0_d[5];
cx s_0_d[0], s_0_d[1];
cx s_0_d[6], s_0_d[3];
cx s_0_d[4], s_0_d[2];
cx s_0_d[6], s_0_d[5];
cx s_0_d[0], s_0_d[3];
cx s_0_d[4], s_0_d[1];
cx s_0_d[3], s_0_d[2];

barrier s_a_0